# 🎙️ Laboratorio Clínico: ShiftSync AI (PoC)
**Objetivo:** Desarrollar una Prueba de Concepto (PoC) para la transcripción y estructuración de entregas de turno de enfermería utilizando IA local.

**Justificación Clínica:** La entrega de turno es un proceso crítico donde la pérdida de información impacta la seguridad del paciente. Usar modelos locales (Whisper + Llama 3) asegura la privacidad de los datos bajo la normativa MINSAL y evita dependencia de internet.

**Arquitectura del Pipeline:**
1. **Audio a Texto:** `Faster-Whisper` (con filtro VAD para ignorar ruido de pasillo).
2. **Texto a Estructura:** Modelo LLM (vía Ollama) calibrado para extracción.
3. **Control de Salida:** Forzado de formato JSON estricto para mapear con la metodología SAER (Situación, Antecedentes, Evaluación, Recomendación).

## ⚙️ Módulo 1: Infraestructura y Entorno (GPU)
Preparación del entorno para garantizar transcripción rápida y privacidad de datos.

In [1]:
import os
import sys

# ==========================================
# 0. Verificación de Seguridad (Requisito de GPU)
# ==========================================
print("🔍 Verificando hardware...")
if os.system("nvidia-smi > /dev/null 2>&1") != 0:
    print("❌ ADVERTENCIA: No se detectó una Tarjeta Gráfica (GPU). La transcripción puede ser lenta.")
else:
    print("✅ GPU detectada correctamente.\n")

# ==========================================
# 1. Instalación de Infraestructura
# ==========================================
!sudo apt-get update -qq && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q -U ollama gradio faster-whisper av

# ==========================================
# 2. Arranque y Descarga de Modelos
# ==========================================
!nohup ollama serve > ollama.log 2>&1 &
!sleep 5
print("\nIniciando descarga de Llama 3...")
!ollama pull llama3
print("\n¡Entorno listo y modelo descargado con éxito!")

## 🗂️ Módulo 2: Configuración Clínica y Plantillas SAER
Aquí definimos el diccionario de plantillas de entrega de turno. Empezaremos con la plantilla SAER estándar (Situación, Antecedentes, Evaluación, Recomendación) aplicable a Unidades Médico-Quirúrgicas y UPC.

In [2]:
# ==========================================
# Módulo 2: Configuración y Reglas Clínicas
# ==========================================

# 1. Variables de Inferencia
MODELO_LLM = "llama3"
TEMPERATURA_LLM = 0.0 # Cero creatividad para evitar alucinaciones en datos clínicos.

# 2. Diccionario de Plantillas de Turno
MIS_PLANTILLAS = {
    "SAER_ESTANDAR": """
1. IDENTIFICACIÓN:
- Cama y Paciente: [X]
- Diagnóstico: [X]
- Alergias: [X]

2. [S] SITUACIÓN (Condición actual y eventos críticos):
- [X]

3. [A] ANTECEDENTES (Comorbilidades relevantes):
- [X]

4. [E] EVALUACIÓN (Estado neurológico, respiratorio, vías y drenajes):
- [X]

5. [R] RECOMENDACIONES (Pendientes de exámenes y fármacos):
- [X]""",

    "URGENCIA_TRIAGE": """
1. IDENTIFICACIÓN BREVE:
- Motivo de Consulta: [X]

2. CATEGORIZACIÓN / ESTADO ACTUAL:
- Signos Vitales Alterados: [X]
- Vías venosas: [X]

3. PLAN / PENDIENTES:
- [X]"""
}

# 3. Reglas Estrictas del Sistema (System Prompt)
REGLAS_SISTEMA = """Eres una enfermera experta en gestión del cuidado. Tu única tarea es escuchar el relato verbal de una entrega de turno y estructurarla estrictamente usando la plantilla SAER proporcionada.

REGLAS INQUEBRANTABLES:
1. RESPONDE ÚNICA Y EXCLUSIVAMENTE CON UN OBJETO JSON VÁLIDO.
2. NO INVENTES DATOS. Si un hallazgo no se menciona en el relato, escribe: "No especificado en entrega verbal."
3. NO RESUMAS NI OMITAS DETALLES. Mantén toda la riqueza clínica del dictado original (cantidades, ubicaciones, observaciones de enfermería), estructurándola dentro de la plantilla. Usa terminología estandarizada pero no recortes el relato.\n",

ESTRUCTURA DEL JSON REQUERIDA:
{
    "metadata_entrega": {
        "cama": "",
        "diagnostico_principal": ""
    },
    "hallazgos_criticos": [],
    "informe_final": "Aquí va el texto completo de la plantilla seleccionada con la información inyectada, manteniendo los saltos de línea (\\n).",
    "advertencias": []
}"""

print("✅ Módulo 2 (Configuración SAER) cargado en memoria.")

## 🎧 Módulo 3: Motor de Procesamiento de Audio (Whisper)
Transcribe el relato de la enfermera utilizando el filtro VAD para ignorar silencios y ruido ambiente.

In [3]:
import time
import librosa
from faster_whisper import WhisperModel

print("⏳ Cargando modelo acústico (Whisper)...")
audio_model = WhisperModel("medium", device="cpu", compute_type="int8")
print("✅ Motor de audio cargado exitosamente.")

def procesar_dictado_audio(ruta_audio: str) -> str:
    if not ruta_audio:
        raise ValueError("No se recibió ningún archivo de audio.")

    print(f"🎙️ Procesando audio: {ruta_audio} ...")
    inicio_ms = time.time()

    # --- BYPASS ANTI-ERRORES ---
    # En lugar de darle la ruta del archivo a Whisper (lo que causa el error de PyAV/metadata),
    # usamos librosa para convertir el audio en un arreglo matemático (NumPy) a 16kHz.
    print("🔄 Decodificando audio de forma segura...")
    audio_array, sr = librosa.load(ruta_audio, sr=16000)

    print("🧠 Transcribiendo (Filtro VAD activado)...")
    segmentos, info = audio_model.transcribe(
        audio_array, # Pasamos el array en lugar de la ruta
        language="es",
        beam_size=5,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=500)
    )

    texto_transcrito = " ".join([segmento.text for segmento in segmentos])
    tiempo_total = round(time.time() - inicio_ms, 2)
    print(f"⏱️ Transcripción completada en {tiempo_total} segundos.")

    return texto_transcrito.strip()


## 🧠 Módulo 4: Extracción de Entidades con LLM
Forzamos al modelo Llama 3 a rellenar la plantilla de turno de enfermería.

In [4]:
import json
import re
import time
import ollama

def extraer_json_seguro(texto_crudo: str) -> dict:
    try:
        match = re.search(r'\{.*\}', texto_crudo, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        return json.loads(texto_crudo)
    except Exception as e:
        print(f"⚠️ Error al parsear JSON: {e}")
        return {"error": "El modelo no devolvió un JSON válido.", "texto_crudo": texto_crudo}

def generar_informe_clinico(transcripcion: str, clave_plantilla: str) -> dict:
    if not transcripcion:
        return {"error": "No hay transcripción para procesar."}

    print("🧠 Llama 3 está estructurando la entrega de turno...")
    inicio_ms = time.time()

    plantilla_texto = MIS_PLANTILLAS.get(clave_plantilla, MIS_PLANTILLAS["SAER_ESTANDAR"])
    prompt_usuario = f"""
    PLANTILLA A UTILIZAR:
    {plantilla_texto}

    ENTREGA DE TURNO (VERBAL):
    {transcripcion}
    """

    try:
        respuesta = ollama.chat(
            model=MODELO_LLM,
            messages=[
                {"role": "system", "content": REGLAS_SISTEMA},
                {"role": "user", "content": prompt_usuario}
            ],
            format='json',
            options={"temperature": TEMPERATURA_LLM}
        )
        texto_respuesta = respuesta['message']['content']
        datos_json = extraer_json_seguro(texto_respuesta)
    except Exception as e:
        datos_json = {"error": f"Fallo en comunicación con Ollama: {str(e)}"}

    print(f"⏱️ Estructuración completada en {round(time.time() - inicio_ms, 2)} seg.")
    return datos_json

## 🔗 Módulo 5: Orquestador

In [5]:
def procesar_caso_completo(ruta_audio: str, clave_plantilla: str) -> dict:
    print(f"\n🚀 Iniciando procesamiento para plantilla: {clave_plantilla}")
    inicio_total = time.time()

    try:
        texto_transcrito = procesar_dictado_audio(ruta_audio)
    except Exception as e:
        return {"ok": False, "error": str(e)}

    if not texto_transcrito:
        return {"ok": False, "error": "Audio vacío."}

    resultado_llm = generar_informe_clinico(texto_transcrito, clave_plantilla)

    if "error" in resultado_llm:
        return {"ok": False, "transcripcion": texto_transcrito, "error": resultado_llm["error"]}

    paquete_final = {
        "ok": True,
        "metodo": "shiftsync_poc_v1",
        "tiempo_procesamiento_seg": round(time.time() - inicio_total, 2),
        "transcripcion": texto_transcrito,
        "metadata_entrega": resultado_llm.get("metadata_entrega", {}),
        "informe_final": resultado_llm.get("informe_final", "Error en formato."),
        "advertencias": resultado_llm.get("advertencias", [])
    }

    return paquete_final

## 🖥️ Módulo 6: Interfaz Gradio (Prueba Rápida)

In [6]:
import gradio as gr
import os

def adaptador_gradio(ruta_audio, clave_plantilla):
    if not ruta_audio:
        return ("⚠️ Sube un audio.", "", {}, None)
    
    resultado = procesar_caso_completo(ruta_audio, clave_plantilla)
    if not resultado.get("ok", False):
        return (f"❌ Error: {resultado.get('error')}", "", resultado, None)
    
    informe = resultado.get("informe_final", "")
    transcripcion = resultado.get("transcripcion", "")
    
    # Generar documento físico (.txt)
    nombre_archivo = "Entrega_Turno_Generada.txt"
    with open(nombre_archivo, "w", encoding="utf-8") as f:
        f.write("=== REPORTE DE ENTREGA DE TURNO ===\n\n")
        f.write(informe)
        f.write("\n\n--- TRANSCRIPCIÓN ORIGINAL ---\n")
        f.write(transcripcion)
        
    return (transcripcion, informe, resultado, nombre_archivo)

with gr.Blocks(title="ShiftSync AI") as demo:
    gr.Markdown("# 🩺 ShiftSync AI - Registro de Entrega de Turno")
    
    with gr.Row():
        with gr.Column(scale=1):
            audio_in = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Relato Verbal")
            plantilla_in = gr.Dropdown(choices=list(MIS_PLANTILLAS.keys()), value="SAER_ESTANDAR", label="Formato Clínico")
            btn_generar = gr.Button("Procesar Turno", variant="primary")
        
        with gr.Column(scale=2):
            transcripcion_out = gr.Textbox(label="Transcripción Cruda", interactive=False)
            informe_out = gr.Textbox(label="Plantilla Completada (Editable)", lines=12)
            json_out = gr.JSON(label="Datos Estructurados para HIS")
            archivo_descarga = gr.File(label="Descargar Documento Generado")
            
    btn_generar.click(
        fn=adaptador_gradio, 
        inputs=[audio_in, plantilla_in], 
        outputs=[transcripcion_out, informe_out, json_out, archivo_descarga]
    )

demo.launch(share=True, debug=True)
